# Thế hệ 6 — Statistical Downscaling / Bias Correction (Quantile Mapping)

Bổ trợ (hoặc thay thế khi không đủ tài nguyên HPC cho RCM) cho thế hệ 5: dùng quan hệ thống kê giữa
phân phối mô hình thô và phân phối quan trắc để sửa lệch (bias correction) — đúng nội dung sách
Hayhoe et al. 2021 và slide Page 2022 (`07_Giao_Trinh_Nen_Tang/Downscaling/`) đã tải về.

Demo: **Quantile Mapping** thật (không giả định tham số phân phối, map theo hàm phân phối tích luỹ
thực nghiệm — đúng kỹ thuật kinh điển trong Maraun & Widmann 2017 / Hayhoe 2021) áp dụng trên dữ liệu
SST thật đã coarsen (thế hệ 3) để sửa lệch về lại phân phối SST thật độ phân giải cao.

In [1]:
DATA_PATH = r"/Users/tuannghiat/Downloads/USTH Phenikaa - Climate Modelling /08_The_He_Mo_Hinh_Thoi_Tiet/_data_cache/oisst_SEA_2020_monthly_sample.nc"
print("Dùng dữ liệu thật:", DATA_PATH)

Dùng dữ liệu thật: /Users/tuannghiat/Downloads/USTH Phenikaa - Climate Modelling /08_The_He_Mo_Hinh_Thoi_Tiet/_data_cache/oisst_SEA_2020_monthly_sample.nc


In [2]:
import xarray as xr
import numpy as np

ds = xr.open_dataset(DATA_PATH)
sst_all = ds['sst'].squeeze()  # (time, lat, lon) 12 tháng thật
sst_coarse_all = sst_all.coarsen(lat=8, lon=8, boundary='trim').mean().interp_like(sst_all, method='linear')

obs = sst_all.values.flatten()
obs = obs[~np.isnan(obs)]
model = sst_coarse_all.values.flatten()
model = model[~np.isnan(model)]

print(f"Phân phối quan trắc thật: mean={obs.mean():.3f}, std={obs.std():.3f}")
print(f"Phân phối mô hình thô (trước sửa lệch): mean={model.mean():.3f}, std={model.std():.3f}")

def quantile_mapping(model_vals, obs_vals, model_sample):
    """Empirical quantile mapping: map từng giá trị model theo quantile của nó trong phân phối model,
    rồi lấy giá trị tương ứng quantile đó trong phân phối obs."""
    model_sorted = np.sort(model_vals)
    obs_sorted = np.sort(obs_vals)
    ranks = np.searchsorted(model_sorted, model_sample) / len(model_sorted)
    ranks = np.clip(ranks, 0, 1)
    idx = (ranks * (len(obs_sorted) - 1)).astype(int)
    return obs_sorted[idx]

corrected = quantile_mapping(model, obs, model)
print(f"Phân phối sau quantile mapping: mean={corrected.mean():.3f}, std={corrected.std():.3f}")
print(f"\nBias trước sửa: {model.mean() - obs.mean():.4f} độ C")
print(f"Bias sau sửa (phải ~0 vì map đúng phân phối chính nó): {corrected.mean() - obs.mean():.4f} độ C")
print("\nLưu ý: đây là self-mapping minh hoạ cơ chế toán học (map model lên đúng phân phối obs nó")
print("được xây từ chính obs này). Ứng dụng thật (vd Hayhoe 2021 ch.5) luôn map model lên obs ĐỘC LẬP")
print("(dùng giai đoạn lịch sử để học hàm map, áp dụng cho giai đoạn tương lai khác).")

Phân phối quan trắc thật: mean=28.179, std=2.318
Phân phối mô hình thô (trước sửa lệch): mean=28.061, std=2.331
Phân phối sau quantile mapping: mean=28.179, std=2.319

Bias trước sửa: -0.1178 độ C
Bias sau sửa (phải ~0 vì map đúng phân phối chính nó): -0.0003 độ C

Lưu ý: đây là self-mapping minh hoạ cơ chế toán học (map model lên đúng phân phối obs nó
được xây từ chính obs này). Ứng dụng thật (vd Hayhoe 2021 ch.5) luôn map model lên obs ĐỘC LẬP
(dùng giai đoạn lịch sử để học hàm map, áp dụng cho giai đoạn tương lai khác).
